# Cognitive Cybersecurity - Protegendo o Futuro do Trabalho Digital

## Autor:

Claudio A. C. Junior

### Introdução

Este notebook tem como objetivo demonstrar, de forma prática, alguns conceitos fundamentais de segurança da informação aplicados a um ambiente que utiliza inteligência artificial e serviços em nuvem. Durante o desenvolvimento, realizamos ações como anonimização de dados, análise de vulnerabilidades conhecidas (CVE), criação de logs, simulação de um ataque tipo ransomware e apresentação de como funciona um portscan.

O foco é mostrar como essas práticas contribuem para proteger aplicações e usuários em um cenário de trabalho híbrido, onde dispositivos pessoais e redes domésticas podem aumentar os riscos de segurança.

In [ ]:
# Importa Path para manipular diretórios e arquivos
from pathlib import Path

# Importa pandas para trabalhar com tabelas e DataFrames
import pandas as pd

# Biblioteca para gerar hashes (usada na anonimização)
import hashlib

# Bibliotecas auxiliares do sistema, JSON e controle de tempo
import os
import json
import time

# Usada para registrar datas e horários (ex.: logs)
from datetime import datetime

# Define o diretório onde serão armazenados os arquivos do projeto
OUT_DIR = Path("/content/gs_cyber_lab")

# Cria o diretório caso ele não exista
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Exibe o caminho do diretório criado
print("Diretório de saída:", OUT_DIR)

Diretório de saída: /content/gs_cyber_lab


In [ ]:
# Cria o DataFrame inicial com dados reais
original = pd.DataFrame({
    "Nome": ["Ana Silva", "João Souza"],
    "Idade": [23, 31],
    "Cidade": ["São Paulo", "Santos"]
})

# Função que aplica SHA-256 em um texto
def sha256_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

# Função que anonimiza uma coluna usando SHA-256
def anonymize_dataframe(df: pd.DataFrame, column: str = "Nome") -> pd.DataFrame:
    df_copy = df.copy()
    df_copy[column] = df_copy[column].astype(str).apply(sha256_hash)
    return df_copy

# Aplica a anonimização aos nomes
anon = anonymize_dataframe(original, "Nome")

# Define caminhos dos arquivos CSV e salva ambos
original_path = OUT_DIR / "dados_antes_anon.csv"
anon_path = OUT_DIR / "dados_depois_anon.csv"
original.to_csv(original_path, index=False, encoding='utf-8')
anon.to_csv(anon_path, index=False, encoding='utf-8')

# Exibe os dados antes e depois da anonimização
print("Dados originais:")
display(original)
print("Dados anonimizados (SHA-256):")
display(anon)

# Mostra onde os arquivos foram salvos
print("Arquivos salvos em:", original_path, anon_path)

Dados originais:


,Nome,Idade,Cidade
0,Ana Silva,23,São Paulo
1,João Souza,31,Santos


Dados anonimizados (SHA-256):


,Nome,Idade,Cidade
0,ad7d6314bec4759aa44e844e7e7e3bfc3c3e27b0d5a3d5...,23,São Paulo
1,6a77047b25e369be3a5ed644ce66fe0d9b31b8b5a23e9a...,31,Santos


Arquivos salvos em: /content/gs_cyber_lab/dados_antes_anon.csv /content/gs_cyber_lab/dados_depois_anon.csv


### Desenvolvimento 1

A primeira etapa do projeto consiste na proteção de dados pessoais através da anonimização. Utilizamos o algoritmo SHA-256, que transforma informações sensíveis, como nomes, em códigos irreversíveis. Essa prática é fundamental em sistemas que lidam com dados pessoais, pois impede que registros possam ser associados diretamente a indivíduos, mesmo em caso de vazamento.

Além disso, a criação dos arquivos CSV antes e depois da anonimização demonstra como o processo pode ser aplicado facilmente no fluxo de tratamento de dados de um sistema real.
Essa etapa reforça princípios da LGPD, garantindo privacidade e segurança no uso de informações para modelos de IA.

In [ ]:
# Cria um DataFrame com vulnerabilidades reais (CVE, CWE e pontuação CVSS)
vulns = pd.DataFrame([
    {"Software": "PostgreSQL", "CVE": "CVE-2024-0985", "CWE": "CWE-89", "CVSS": 7.3, "Resumo": "Possibilidade de SQL Injection"},
    {"Software": "Apache HTTP Server", "CVE": "CVE-2023-25690", "CWE": "CWE-20", "CVSS": 7.5, "Resumo": "Falha em validação de entrada"},
    {"Software": "Node.js", "CVE": "CVE-2024-22020", "CWE": "CWE-918", "CVSS": 8.1, "Resumo": "Risco de SSRF (requisições internas)"}
])

# Define o caminho onde o CSV será salvo
vuln_path = OUT_DIR / "vulnerabilidades.csv"

# Salva a tabela de vulnerabilidades em um arquivo CSV
vulns.to_csv(vuln_path, index=False, encoding='utf-8')

# Exibe o DataFrame com as vulnerabilidades
print("Vulnerabilidades (exibindo tabela):")
display(vulns)

# Informa o caminho do arquivo gerado
print("CSV gerado em:", vuln_path)

Vulnerabilidades (exibindo tabela):


,Software,CVE,CWE,CVSS,Resumo
0,PostgreSQL,CVE-2024-0985,CWE-89,7.3,Possibilidade de SQL Injection
1,Apache HTTP Server,CVE-2023-25690,CWE-20,7.5,Falha em validação de entrada
2,Node.js,CVE-2024-22020,CWE-918,8.1,Risco de SSRF (requisições internas)


CSV gerado em: /content/gs_cyber_lab/vulnerabilidades.csv


In [ ]:
# Guarda o código Python do exemplo de rate limiting em formato de string
flask_code = """
# Arquivo: rate_limit_example.py
# Exemplo básico de rate-limiting em Flask (educacional).
from flask import Flask, request, jsonify
import time

app = Flask(__name__)

REQUESTS = {}
MAX_PER_MIN = 5  # limite por IP por minuto

def cleanup(ip):
    now = time.time()
    REQUESTS[ip] = [t for t in REQUESTS.get(ip, []) if now - t < 60]

@app.route('/api', methods=['GET'])
def api_root():
    ip = request.remote_addr or 'unknown'
    cleanup(ip)
    if len(REQUESTS.get(ip, [])) >= MAX_PER_MIN:
        return jsonify({'error':'Too many requests'}), 429
    REQUESTS.setdefault(ip, []).append(time.time())
    return jsonify({'message':'Requisição aceita'})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
"""

# Define o caminho onde o arquivo do código será salvo
rate_file = OUT_DIR / "rate_limit_example.py"

# Salva o código acima em um arquivo .py
rate_file.write_text(flask_code, encoding='utf-8')

# Informa ao aluno onde o arquivo foi criado
print("Arquivo de exemplo de rate limiting salvo em:", rate_file)

# Explica como testar o arquivo localmente
print("Dica: para testar, copie este arquivo para seu PC/VM e execute: python rate_limit_example.py")


Arquivo de exemplo de rate limiting salvo em: /content/gs_cyber_lab/rate_limit_example.py
Dica: para testar, copie este arquivo para seu PC/VM e execute: python rate_limit_example.py


### Desenvolvimento 2

Nesta etapa, ampliamos a análise de segurança para outras áreas: vulnerabilidades conhecidas, controle de requisições e simulação de ataques. Na parte de vulnerabilidades, trabalhamos com uma tabela de CVE reais, representando riscos comuns em softwares amplamente utilizados. Isso reforça a importância de atualizações, patches e monitoramento constante.

Também implementamos um exemplo de rate limiting, que é essencial para evitar abusos na API, como ataques de força bruta ou sobrecarga. Em seguida, realizamos a simulação de um comportamento típico de ransomware, renomeando arquivos para demonstrar o impacto de um ataque e a importância de backups e planos de recuperação.

Com isso, demonstramos na prática como múltiplas camadas de segurança podem trabalhar juntas para proteger um sistema moderno.

In [ ]:
# Define a pasta onde ficará o arquivo simulado do "modelo"
demo_dir = OUT_DIR / "demo_model"
demo_dir.mkdir(exist_ok=True)

# Cria o arquivo de modelo fictício (dummy) caso ainda não exista
model_file = demo_dir / "modelo.pkl"
if not model_file.exists():
    model_file.write_text("conteudo dummy do modelo", encoding='utf-8')
    print("Arquivo de modelo criado em:", model_file)

# Função que simula o ataque: renomeia arquivos adicionando ".lock"
def simulate_ransomware_lock(target_files):
    locked = []
    for f in target_files:
        if f.exists():
            locked_name = f.with_suffix(f.suffix + ".lock")
            f.rename(locked_name)
            locked.append(locked_name)
    return locked

# Função que restaura os arquivos removendo o sufixo ".lock"
def simulate_ransomware_restore(locked_files):
    restored = []
    for f in locked_files:
        if f.exists() and f.suffix.endswith(".lock"):
            original_name = Path(str(f)[:-5])  # remove o ".lock"
            f.rename(original_name)
            restored.append(original_name)
    return restored

# Executa a simulação de bloqueio dos arquivos
locked = simulate_ransomware_lock([model_file])
print("Estado após 'lock' (arquivos renomeados):")
for p in demo_dir.glob("*"):
    print("-", p.name)

# Executa a restauração dos arquivos bloqueados
restored = simulate_ransomware_restore(locked)
print("Estado após 'restore':")
for p in demo_dir.glob("*"):
    print("-", p.name)

Arquivo de modelo criado em: /content/gs_cyber_lab/demo_model/modelo.pkl
Estado após 'lock' (arquivos renomeados):
- modelo.pkl.lock
Estado após 'restore':
- modelo.pkl


In [ ]:
# Define o caminho onde o arquivo de logs será salvo (formato JSONL)
log_path = OUT_DIR / "example_logs.jsonl"

# Cria uma lista de eventos simulando atividades do sistema
logs = [
    {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "login_success", "user":"ana.s", "ip":"192.0.2.10"},
    {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "api_request", "path":"/predict", "ip":"192.0.2.10"},
    {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "failed_login", "user":"joao.s", "ip":"198.51.100.22"},
    {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "rate_limit_block", "ip":"198.51.100.22"}
]

# Abre o arquivo e grava cada log em uma linha (formato JSONL)
with open(log_path, "w", encoding="utf-8") as f:
    for entry in logs:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# Mostra onde o arquivo foi salvo
print("Logs salvos em:", log_path)

# Lê o arquivo e exibe todos os logs registrados
with open(log_path, "r", encoding="utf-8") as f:
    print(f.read())

Logs salvos em: /content/gs_cyber_lab/example_logs.jsonl
{"timestamp": "2025-11-19T13:59:13.393254Z", "event": "login_success", "user": "ana.s", "ip": "192.0.2.10"}
{"timestamp": "2025-11-19T13:59:13.393317Z", "event": "api_request", "path": "/predict", "ip": "192.0.2.10"}
{"timestamp": "2025-11-19T13:59:13.393344Z", "event": "failed_login", "user": "joao.s", "ip": "198.51.100.22"}
{"timestamp": "2025-11-19T13:59:13.393365Z", "event": "rate_limit_block", "ip": "198.51.100.22"}



/tmp/ipython-input-675213536.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "login_success", "user":"ana.s", "ip":"192.0.2.10"},
/tmp/ipython-input-675213536.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"timestamp": datetime.utcnow().isoformat()+"Z", "event": "api_request", "path":"/predict", "ip":"192.0.2.10"},
/tmp/ipython-input-675213536.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  {"timestamp": datetime.utcnow().isoformat()+"Z", "event":

In [ ]:
# Função que cria um "resultado falso" de portscan apenas para demonstração
def mocked_portscan_report(target_ip: str):
    return {
        "target": target_ip,
        "open_ports": [
            {"port":22, "service":"ssh", "version":"OpenSSH 7.2p2"},
            {"port":80, "service":"http", "version":"Apache httpd 2.4.18"},
            {"port":443, "service":"https", "version":"nginx 1.14.0"}
        ],
        # Aviso explicando que isso é apenas um exemplo, não um scan real
        "note": "Resultado mockado para demonstração. Rode `nmap -sV {target}` em sua máquina para saída real."
    }

# Gera o resultado simulado para o IP escolhido
mock = mocked_portscan_report("192.168.0.10")

# Importa pprint para exibir o dicionário de forma organizada
import pprint

# Mostra o resultado mockado do portscan
pprint.pprint(mock)

{'note': 'Resultado mockado para demonstração. Rode `nmap -sV {target}` em sua '
         'máquina para saída real.',
 'open_ports': [{'port': 22, 'service': 'ssh', 'version': 'OpenSSH 7.2p2'},
                {'port': 80,
                 'service': 'http',
                 'version': 'Apache httpd 2.4.18'},
                {'port': 443, 'service': 'https', 'version': 'nginx 1.14.0'}],
 'target': '192.168.0.10'}


In [ ]:
# Cria um dicionário com os caminhos dos arquivos gerados ao longo do projeto
report = {
    "anonymization": {
        "original_csv": str(OUT_DIR / "dados_antes_anon.csv"),
        "anon_csv": str(OUT_DIR / "dados_depois_anon.csv")
    },
    "vulnerabilities_csv": str(OUT_DIR / "vulnerabilidades.csv"),
    "rate_limit_example": str(OUT_DIR / "rate_limit_example.py"),
    "demo_model_dir": str(OUT_DIR / "demo_model"),
    "logs": str(OUT_DIR / "example_logs.jsonl"),
    "mocked_portscan": mock
}

# Define o caminho do arquivo onde o resumo será salvo
report_path = OUT_DIR / "lab_report_summary.json"

# Escreve o arquivo JSON com todas as informações acima
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

# Mostra ao usuário onde o arquivo foi salvo
print("Resumo do relatório salvo em:", report_path)

# Orienta como baixar os arquivos no Colab
print("Você pode baixar os arquivos a partir do painel do Colab (Files) ou usar files.download() para cada um.")

Resumo do relatório salvo em: /content/gs_cyber_lab/lab_report_summary.json
Você pode baixar os arquivos a partir do painel do Colab (Files) ou usar files.download() para cada um.


### Conclusão

O trabalho permitiu aplicar, de forma prática, conceitos essenciais de segurança da informação dentro de um ambiente que utiliza inteligência artificial e serviços em nuvem. As etapas realizadas, anonimização, análise de vulnerabilidades, geração de logs, simulação de ransomware e demonstração de portscan, mostram que é possível criar uma base sólida de proteção mesmo em sistemas simples.

Além disso, foi reforçada a importância de boas práticas, como controle de acesso, criptografia, monitoramento contínuo e políticas de segurança. Ao compreender essas técnicas, torna-se mais fácil desenvolver aplicações mais seguras e resilientes, capazes de proteger dados sensíveis e garantir a integridade do sistema em um cenário de trabalho híbrido.